In [ ]:
!pip install -U diffusers transformers accelerate torch torchvision -q

In [ ]:
!pip uninstall -y torchaudio -q

In [ ]:
import torch, gc, json, requests
from diffusers import FluxPipeline, StableDiffusionXLPipeline, StableDiffusion3Pipeline

url = "https://raw.githubusercontent.com/maram-elaian/brandora/main/data/test_briefs.json"
briefs = requests.get(url).json()[:5]   # 5 بس للتجربة الأولى، مش الـ30 كاملة

def brief_to_image_prompt(b):
    return f"{b['logo_direction']}, {', '.join(b['visual_style'])} style, vector logo, clean background, no text"

models_to_check = [
    {"name": "FLUX_schnell", "id": "black-forest-labs/FLUX.1-schnell", "cls": FluxPipeline, "dtype": torch.bfloat16, "steps": 4, "guidance": 0.0},
    {"name": "Playground_v2.5", "id": "playgroundai/playground-v2.5-1024px-aesthetic", "cls": StableDiffusionXLPipeline, "dtype": torch.float16, "steps": 25, "guidance": 3.0},
    {"name": "SD3_medium", "id": "stabilityai/stable-diffusion-3-medium-diffusers", "cls": StableDiffusion3Pipeline, "dtype": torch.float16, "steps": 28, "guidance": 7.0},
]

for m in models_to_check:
    print(f"⏳ جاري تحميل: {m['name']}")
    pipe = None
    try:
        pipe = m["cls"].from_pretrained(m["id"], torch_dtype=m["dtype"])
        pipe.enable_model_cpu_offload()

        for brief in briefs:
            prompt = brief_to_image_prompt(brief)
            image = pipe(
                prompt,
                num_inference_steps=m["steps"],
                guidance_scale=m["guidance"],
            ).images[0]
            filename = f"{m['name']}_{brief['id']}.png"
            image.save(filename)
            print(f"   ✅ {brief['id']} → {filename}")

    except Exception as e:
        print(f"   ❌ فشل! {type(e).__name__}: {e}")

    finally:
        if pipe is not None:
            del pipe
        gc.collect()
        torch.cuda.empty_cache()

print("✅ انتهى!")